# Adapter Smoke Test

Directly load a trained continual SD-LoRA adapter on top of the DINO backbone, inspect its metadata, run one prediction, and optionally sanity-check a small folder of images.

This notebook now scans the project folder and common Drive roots for adapter bundles, then lets you choose which one to load.

Manual path inputs are accepted too:
- `ADAPTER_DIR`: adapter folder, parent export folder, telemetry run folder, telemetry `artifacts/` folder, or `adapter_meta.json`
- `ADAPTER_ROOT`: parent of crop folders such as `models/adapters/`
- `IMAGE_PATH`: one image file
- `BATCH_IMAGE_DIR`: one folder containing image files

Retry controls for the same session:
- set `FORCE_ADAPTER_RESCAN = True` and rerun the adapter discovery cell to rebuild the list
- set `FORCE_IMAGE_UPLOAD = True` and rerun the single-image cell to upload a different image

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def refresh_colab_clone(repo_root: Path) -> None:
    repo_root = Path(repo_root)
    if repo_root != CLONE_TARGET or not (repo_root / '.git').exists():
        return
    try:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', REPO_URL], check=False)
        subprocess.run(['git', '-C', str(repo_root), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'checkout', 'master'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only', 'origin', 'master'], check=True)
        print(f'Refreshed Colab clone at {repo_root}')
    except Exception as exc:
        print(f'Warning: could not refresh Colab clone at {repo_root}: {exc}')

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab
except ModuleNotFoundError:
    if not (CLONE_TARGET / 'scripts').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    os.chdir(CLONE_TARGET)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    refresh_colab_clone(ROOT)
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)

In [ ]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import (
    discover_adapter_candidates,
    load_adapter_summary,
    predict_image_folder,
    predict_single_image,
)

CROP_NAME = None
# Optional crop filter, e.g. 'tomato'. Leave as None to show all discovered adapters.

ADAPTER_DIR = None
# Optional manual override. Accepted values include:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ or .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Fallback deployed root. This should normally be the parent of crop folders, e.g. models/adapters/.

SEARCH_ROOTS = [
    ROOT,
    ROOT / 'outputs',
    ROOT / 'models',
    ROOT / 'models' / 'adapters',
    Path('/content/drive/MyDrive/aads_ulora'),
]
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# One image file path such as /content/sample_leaf.jpg or D:/data/sample_leaf.png
BATCH_IMAGE_DIR = None
# One folder path containing image files for the optional batch sanity check
DEVICE = 'cuda'

print('Manual ADAPTER_DIR overrides discovery. Otherwise the notebook scans SEARCH_ROOTS and lists every adapter it finds.')
print('Discovery roots:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('ADAPTER_ROOT should be the parent of crop folders, IMAGE_PATH should be one file, and BATCH_IMAGE_DIR should be one folder.')
print('Set FORCE_ADAPTER_RESCAN=True to rebuild the adapter list, or FORCE_IMAGE_UPLOAD=True to upload a new image without restarting the notebook.')

In [ ]:
adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, rebuilding adapter discovery list.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Using manual ADAPTER_DIR: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'No adapters found under SEARCH_ROOTS. Set ADAPTER_DIR manually to an adapter folder, telemetry run folder, export folder, or adapter_meta.json, or update SEARCH_ROOTS.'
        )

    print('Discovered adapters:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Choose an adapter in the dropdown, then run the next cell.')
    else:
        print('ipywidgets unavailable. Set SELECTED_ADAPTER_INDEX to the adapter you want, then run the next cell.')

In [ ]:
if ADAPTER_DIR is None:
    selected_index = adapter_selector.value if adapter_selector is not None else SELECTED_ADAPTER_INDEX
    selected_candidate = adapter_candidates[int(selected_index)]
    ADAPTER_DIR = selected_candidate['adapter_dir']
    if CROP_NAME is None:
        CROP_NAME = selected_candidate.get('crop_name')
    print(json.dumps(selected_candidate, indent=2))
    FORCE_ADAPTER_RESCAN = False

summary = load_adapter_summary(
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env='colab',
    device=DEVICE,
)

CROP_NAME = summary['crop_name']
ADAPTER_DIR = summary['resolved_adapter_dir']
print(f"Resolved crop_name={CROP_NAME}")
print(f"Resolved adapter_dir={ADAPTER_DIR}")

print(json.dumps(summary, indent=2))

In [ ]:
if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, requesting a new image upload.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Set IMAGE_PATH to one image file when running outside Colab, for example D:/data/leaf.jpg.')
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Using IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env='colab',
    device=DEVICE,
)

print(json.dumps(single_result, indent=2))

In [ ]:
if not BATCH_IMAGE_DIR:
    print('Set BATCH_IMAGE_DIR to a folder containing image files to run the optional folder sanity check.')
else:
    rows = predict_image_folder(
        BATCH_IMAGE_DIR,
        CROP_NAME,
        adapter_dir=ADAPTER_DIR,
        adapter_root=ADAPTER_ROOT,
        config_env='colab',
        device=DEVICE,
    )
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
    else:
        print(rows)

    predicted_counts = Counter(row['predicted_class'] for row in rows if row.get('predicted_class'))
    ood_count = sum(1 for row in rows if row.get('is_ood') is True)
    error_count = sum(1 for row in rows if row.get('error'))

    print('predicted_class_counts:', dict(predicted_counts))
    print('ood_count:', ood_count)
    print('error_count:', error_count)